In [0]:
silver_path = "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/silver/"

In [0]:
from pyspark.sql.functions import col

df_emp_clean = spark.read.parquet(
    "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/employees"
).withColumn("join_date", col("join_date").cast("Date"))

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5
4,David,HR,61000,2022-01-10,M,27,true,6100,4
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3


In [0]:
from pyspark.sql.functions import year, when, col

df_emp_clean = df_emp_clean.withColumn("join_year", year(col("join_date")))

df_emp_clean = df_emp_clean.withColumn(
    "experience_level", 
    when(col("join_year") < 2019, "Senior")
    .when((col("join_year") >= 2019) & (col("join_year") <= 2021), "Mid")
    .otherwise("Junior")
).drop("join_year")  

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4,Mid
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3,Mid
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5,Mid
4,David,HR,61000,2022-01-10,M,27,true,6100,4,Junior
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4,Senior
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5,Senior
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3,Junior


In [0]:
from pyspark.sql.functions import col, datediff, lit, to_date, current_timestamp

df_emp_clean = df_emp_clean.withColumn("days_employed", datediff(to_date(current_timestamp(), "yyyy-MM-dd"), col("join_date")))

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level,days_employed
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4,Mid,1874
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3,Mid,2497
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5,Mid,1989
4,David,HR,61000,2022-01-10,M,27,true,6100,4,Junior,1573
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4,Senior,2899
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5,Senior,3136
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3,Junior,1173


In [0]:
from pyspark.sql.functions import year, when, col, datediff, lit, to_date, current_timestamp

df_emp_clean = df_emp_clean.withColumn(
    "salary_band",
    when(col("salary") >= 90000, "High")
    .when(col("salary") >= 70000, "Mid")
    .otherwise("Low")
)

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level,days_employed,salary_band
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4,Mid,1874,High
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3,Mid,2497,Mid
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5,Mid,1989,Mid
4,David,HR,61000,2022-01-10,M,27,true,6100,4,Junior,1573,Low
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4,Senior,2899,Mid
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5,Senior,3136,High
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3,Junior,1173,Low


In [0]:
df_emp_clean = df_emp_clean.filter(col("is_active")== "True")

display(df_emp_clean)

id,name,dept,salary,join_date,gender,age,is_active,bonus,rating,experience_level,days_employed,salary_band
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4,Mid,1874,High
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3,Mid,2497,Mid
4,David,HR,61000,2022-01-10,M,27,true,6100,4,Junior,1573,Low
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4,Senior,2899,Mid
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3,Junior,1173,Low


In [0]:
df_emp_clean = df_emp_clean.withColumnRenamed("dept", "department")
df_emp_clean = df_emp_clean.drop("gender")

display(df_emp_clean)

id,name,department,salary,join_date,age,is_active,bonus,rating,experience_level,days_employed,salary_band
1,Alice,Engineering,95000,2021-03-15,29,true,9500,4,Mid,1874,High
2,Bob,Marketing,72000,2019-07-01,35,true,7200,3,Mid,2497,Mid
4,David,HR,61000,2022-01-10,27,true,6100,4,Junior,1573,Low
5,Eve,Marketing,79000,2018-05-25,38,true,7900,4,Senior,2899,Mid
7,Grace,HR,58000,2023-02-14,24,true,5800,3,Junior,1173,Low


In [0]:
df_emp_clean.write.mode("overwrite").parquet(silver_path + "employees_clean")